In [ ]:
# Cell 1: Imports
import pyfsntfs
import csv
from datetime import datetime, timedelta
from pathlib import Path

# --- Paths --- #
LOGFILE_PATH = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/training/logfile/logfile raw/01-PE-LogFile")
OUTPUT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/training/logfile/logfile parsed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "01-PE-LogFile.csv"

# --- Constants --- #
EPOCH_DT = datetime(1601, 1, 1)  # FILETIME epoch

# --- Helper: FILETIME to datetime + nanoseconds --- #
def filetime_to_ns(dt_filetime):
    """Convert FILETIME (100ns since 1601-01-01) to datetime and nanosecond string"""
    if dt_filetime is None:
        return (None, "")
    try:
        # Convert to datetime
        dt = EPOCH_DT + timedelta(microseconds=dt_filetime // 10)
        # Calculate total nanoseconds
        total_ns = dt_filetime * 100
        base = dt.strftime('%Y-%m-%dT%H:%M:%S')
        decimal_part = f"{total_ns % 1_000_000_000:09d}"[:7]  # preserve nanosecond precision
        return (dt, f"{base}.{decimal_part}Z")
    except Exception:
        return (None, "")


In [ ]:
# --- Cell 2: Parse $LogFile and output CSV --- #
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "LSN",
        "EventTime",
        "Redo_OP",
        "Undo_OP",
        "Record_Offset",
        "Attribute_Offset",
        "RedoData_raw",
        "UndoData_raw",
        "CreationTime_Redo_dt",
        "ModificationTime_Redo_dt",
        "AccessTime_Redo_dt",
        "EntryTime_Redo_dt",
        "CreationTime_Redo_ns",
        "ModificationTime_Redo_ns",
        "AccessTime_Redo_ns",
        "EntryTime_Redo_ns",
        "CreationTime_Undo_dt",
        "ModificationTime_Undo_dt",
        "AccessTime_Undo_dt",
        "EntryTime_Undo_dt",
        "CreationTime_Undo_ns",
        "ModificationTime_Undo_ns",
        "AccessTime_Undo_ns",
        "EntryTime_Undo_ns"
    ])

    # Open the logfile
    with pyfsntfs.volume() as vol:
        vol.open(str(LOGFILE_PATH))
        
        for record in vol.logfile_records():
            redo = record.redo_data
            undo = record.undo_data

            # Extract timestamps from redo and undo (FILETIME)
            redo_ts = getattr(redo, 'timestamps', [None]*4)
            undo_ts = getattr(undo, 'timestamps', [None]*4)

            # Convert to datetime + nanoseconds
            redo_dt_ns = [filetime_to_ns(ft) for ft in redo_ts]
            undo_dt_ns = [filetime_to_ns(ft) for ft in undo_ts]

            # Split into separate columns
            redo_dt = [x[0] for x in redo_dt_ns]
            redo_ns = [x[1] for x in redo_dt_ns]
            undo_dt = [x[0] for x in undo_dt_ns]
            undo_ns = [x[1] for x in undo_dt_ns]

            # Write CSV row
            writer.writerow([
                record.lsn,
                record.event_time,
                record.redo_op,
                record.undo_op,
                record.record_offset,
                record.attribute_offset,
                redo.raw.hex() if redo else "",
                undo.raw.hex() if undo else "",
                *redo_dt,
                *redo_ns,
                *undo_dt,
                *undo_ns
            ])

print(f"✓ $LogFile parsing complete with nanosecond preservation. CSV saved at: {OUTPUT_CSV}")
